# Analytics Example - Spark Routes - Contract Month Seasonality

This script allows you to plot the price evolution & seasonality for a given route and month. In this case the route chosen is Sabine Pass to Futtsu via COGH.

For a full explanation of how to import our Freight Routes data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/lng-freight/routes.html

__N.B. This script requires a Freight subscription__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

In [ ]:
# Define function for listing routes from API
def list_routes(access_token):
    """
    Fetch available routes. Return route ids and Spark price release dates.

    The URI used returns a list of all available Spark routes. With these routes, we can find the price breakdown of a specific route.

    # Procedure:

    Do a GET query to /v1.0/routes/ with a Bearer token authorization HTTP header.
    """
    content = do_api_get_query(uri="/v1.0/routes/", access_token=access_token)

    print(">>>> All the routes you can fetch")
    tickers = []
    for contract in content["data"]["routes"]:
        tickers.append(contract["uuid"])

    reldates = content["data"]["sparkReleaseDates"]

    dicto1 = content["data"]

    return tickers, reldates, dicto1

### N.B. Credentials

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already. Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html


In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## 2. Routes reference data

In [ ]:
## Define function for listing routes reference data

def list_routes_refdata(access_token):

    content = do_api_get_query(uri="/v1.0/routes/", access_token=access_token)

    reldates = content["data"]["sparkReleaseDates"]

    data_dict = content["data"]

    return reldates, data_dict


# Fetch all reference data
reldates, refdata_dict = list_routes_refdata(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['routes'])
available_df = available_df.rename(columns={
    'uuid' : 'UUID',
    'via' : 'Via',
    'loadPort.uuid' : 'LoadPortUUID',
    'loadPort.type' : 'LoadPortType',
    'loadPort.region' : 'LoadPortRegion',
    'loadPort.name' : 'LoadPortName',
    'dischargePort.uuid' : 'DischargePortUUID',
    'dischargePort.type' : 'DischargePortType',
    'dischargePort.region' : 'DischargePortRegion',
    'dischargePort.name' : 'DischargePortName',
})

available_df.head(3)

# 3. Defining Input Route Parameters

In [ ]:
## Calling that function and storing the output

# Here we store the entire dataset called from the API

load = 'Sabine Pass'
discharge = 'Futtsu'
via = 'cogh'

my_route = available_df[(available_df["LoadPortName"] == load) & \
                    (available_df["DischargePortName"] == discharge) & \
                    (available_df['Via'] == via)]['UUID'].values[0]


# 4. Fetching Routes Data

In [ ]:
import time

## Defining the function to fetch Routes data

def fetch_route_data(access_token, ticker, release, congestion_laden= None, congestion_ballast= None):

    query_params = "?release-date={}".format(release)
    if congestion_laden is not None:
        query_params += "&congestion-laden-days={}".format(congestion_laden)
    if congestion_ballast is not None:
        query_params += "&congestion-ballast-days={}".format(congestion_ballast)

    uri = "/v1.0/routes/{}/{}".format(ticker, query_params)
    #print(uri)

    content = do_api_get_query(
        uri="/v1.0/routes/{}/{}".format(ticker, query_params),
        access_token=access_token,
    )

    my_dict = content["data"]

    return my_dict

## Defining function to fetch historical data for a range of release dates & store as Dataframe

def routes_history(route_uuid, reldates, laden=None, ballast=None):
    
    # initiliasing data dictionary
    my_route  = {
            "ReleaseDate": [],
            "Period": [],
            "CalMonth":[],
            "StartDate": [],
            "EndDate": [],
            "Total": [],
            "Hire": [],
            "Fuel": [],
            "Port": [],
            "Canal": [],
            "Carbon": [],
            "Congestion": [],
        }

    # iterating through release dates
    for r in reldates:
        try:
            my_dict = fetch_route_data(access_token, route_uuid, release=r,  congestion_laden=laden, congestion_ballast=ballast)
        except:
            print('Bad Date: ' + r)
            continue
        
        # Appending all data for full forward curve per release date
        for data in my_dict["dataPoints"]:
            my_route['StartDate'].append(data["deliveryPeriod"]["startAt"])
            my_route['EndDate'].append(data["deliveryPeriod"]["endAt"])
            my_route['Period'].append(data["deliveryPeriod"]["name"])

            my_route['Total'].append(data["costsInUsdPerMmbtu"]["total"])
            my_route['Hire'].append(data["costsInUsdPerMmbtu"]["hire"])
            my_route['Fuel'].append(data["costsInUsdPerMmbtu"]["fuel"])
            my_route['Port'].append(data["costsInUsdPerMmbtu"]["port"])
            my_route['Canal'].append(data["costsInUsdPerMmbtu"]["canal"])
            
            # Add congestion and/or carbon costs if applicable
            try:
                my_route['Congestion'].append(data["costsInUsdPerMmbtu"]["congestion"])
            except:
                 my_route['Congestion'].append(0)
            
            try:
                my_route['Carbon'].append(data["costsInUsdPerMmbtu"]["carbon"])
            except:
                 my_route['Carbon'].append(0)

            my_route['ReleaseDate'].append(r)
            my_route['CalMonth'].append(datetime.datetime.strptime(data["deliveryPeriod"]["startAt"], '%Y-%m-%d').strftime('%b%y'))

        # Sleep function is used to avoid API rate limiting.
        time.sleep(0.2)
    
    # convert into Pandas dataframe
    my_route_df = pd.DataFrame(my_route)

    ## Changing the data types to pandas datetime
    for col in ['ReleaseDate', 'StartDate', 'EndDate']:
        my_route_df[col] = pd.to_datetime(my_route_df[col])

    # sorting data by Release Date (latest first)
    my_route_df = my_route_df.sort_values('ReleaseDate')
    
    ## Changing the data type of these columns from 'string' to numbers.
    for col in ["Total","Hire","Fuel","Port","Canal","Congestion", "Carbon"]:
        my_route_df[col] = pd.to_numeric(my_route_df[col])
    
    return my_route_df





In [ ]:
# Calling function to get historical data
# Here we call all historical data
histdf = routes_history(my_route, reldates[:], laden =None, ballast=None)

In [ ]:
# Preview of the data
histdf.head(5)

In [ ]:
# Define functon to calculate the "Day of Year" number for each datapoint. 
# Y-1 datapoints are offset by 365 days, and Y-2 by 730 days, so that timeseries can be plotted in a single, trackable graph.
def sort_years(df):
    
    reldates = df['ReleaseDate'].to_list()
    startdates = df['StartDate'].to_list()
    calmonths = df['CalMonth'].to_list()

    dayofyear = []
    
    for r in reldates:
        ir = reldates.index(r)
        if r.year - startdates[ir].year == -1:
            dayofyear.append(r.timetuple().tm_yday - 365)
        elif r.year - startdates[ir].year == -2:
            dayofyear.append(r.timetuple().tm_yday - 730)
        else:
            dayofyear.append(r.timetuple().tm_yday)
            
    df['DayofYear'] = dayofyear

    return df

histdf = sort_years(histdf)

## Group Dataframe by CalMonth and choose contracts to compare

In [ ]:
# Group DataFrame by Contract Month
hist_months = histdf[(~histdf['Period'].str.contains('Cal', na=False)) & \
                    (histdf['Period'] != 'Spot (Physical)')].copy()
groups = hist_months.groupby('CalMonth')
years = list(hist_months['ReleaseDate'].dt.year.unique())

### Input which month you'd like to compare contracts for

In [ ]:
month = 'Dec'

## 4. Plotting

In [ ]:
## Plotting
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style(style="whitegrid")

fig, ax = plt.subplots(figsize=(15, 7))

max_dates = []

# iterating through years and plotting the data
for y in years:
    ydf = groups.get_group(month + str(y)[2:])
    ydf = sort_years(ydf)
    ax.plot(ydf["DayofYear"], ydf["Total"], label=month + str(y))
    max_dates.append(ydf["DayofYear"].max())

plt.xlabel("Release Date")
plt.ylabel("Cost in USD/MMBtu")

# Setting custom x-axis ticks location and labels.
#xlabels = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'Year End']
xlabels = ['Y-1', 'March', 'May', 'July', 'September', 'November', 'Y+0', 'March', 'May', 'July', 'September', 'November', 'Year End']

# xpos gives the first day of every other month in terms of 'DayofYear'
#xpos = [0,32,60,91,121,152,182,213,244,274,305,335, 365]
xpos = [-365,-305,-244,-183,-121,-60,0,60,121,182,244,305,365]

current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['$ {:,.0f}'.format(x) for x in current_values])

plt.xticks(xpos, xlabels)

plt.xlim(-365,max(max_dates))

# Setting the graph legend
plt.legend()
